<div dir="rtl" lang="he" align="right" markdown="1">

# איך יודעים שזה עובד

מערכת `RAG` היא שני חצאים. אחד מוצא מסמכים, השני כותב תשובה. כשהתשובה יוצאת גרועה, השאלה הראשונה היא איזה מהחצאים אשם — ואי אפשר לענות עליה אם מודדים רק את הסוף.

החצי הראשון קל למדידה: זה חשבון מדויק על רשימה מדורגת, בלי שום מודל. החצי השני קשה, ושם נכנס `LLM-as-judge`.

**מה שיקרה בפרק הזה:** נבנה 12 שאלות שלכל אחת שתי תשובות — אחת שמכילה את העובדה ואחת שלא. מה שנכון ידוע מראש, ולכן אפשר למדוד את השופט עצמו. הוא ייכשל, והכישלון שלו הוא התוכן.

</div>

In [ ]:
import json
from pathlib import Path
from statistics import mean

from aihe import viz
from aihe.judge import (
    agreement,
    build_cases,
    contains_fact,
    holistic_summary,
    judge_cases,
)
from aihe.metrics import mrr, ndcg_at_k, precision_at_k, recall_at_k
from aihe.models import asker

HERE = Path("data") if Path("data/judge-cases.json").exists() else Path("chapters/03-evals/data")
CASSETTES = HERE.parent / "cassettes"
raw = json.loads((HERE / "judge-cases.json").read_text(encoding="utf-8"))["cases"]
cases = build_cases(raw)
print(len(raw), "questions ->", len(cases), "answers to judge")

<div dir="rtl" lang="he" align="right" markdown="1">

## החצי הקל: למדוד אחזור

ארבעה מדדים, וההבדל ביניהם הוא מה שכל אחד לא רואה. נבחן אותם על רשימה קטנה שאפשר לבדוק ביד: חמישה מסמכים, והמסמך הנכון הוא `7`.

שימו לב במיוחד לשורה האחרונה. `recall` לא מבחין בין מסמך שחזר ראשון למסמך שחזר שלישי, ו-`nDCG` כן. זה בדיוק ההבדל שהסביר בפרק 01 למה ה-`reranker` שיפר את `recall@1` ולא נגע ב-`recall@3`.

</div>

In [ ]:
early = [7, 1, 2, 3, 4]   # the right document came back first
late  = [1, 2, 7, 3, 4]   # the same document, two places lower
relevant = {7}

for name, ranking in (("first", early), ("third", late)):
    print(f"  {name:6s} recall@3={recall_at_k(ranking, relevant, 3):.2f}  "
          f"precision@3={precision_at_k(ranking, relevant, 3):.2f}  "
          f"nDCG@3={ndcg_at_k(ranking, relevant, 3):.3f}  "
          f"MRR={mrr([(ranking, relevant)]):.3f}")

<div dir="rtl" lang="he" align="right" markdown="1">

## החצי הקשה: למדוד תשובה

כאן אין רשימה מדורגת ואין תשובה נכונה אחת. הדרך הרווחת היא לשאול מודל אחר, וזה נקרא `LLM-as-judge`.

כדי לבדוק אם זה עובד צריך משהו שאי אפשר להתווכח איתו. לכן כל שאלה כאן מגיעה עם שתי תשובות שנכתבו מראש: אחת אומרת את העובדה — "שישים דקות" — והשנייה שוטפת, עניינית, נשמעת מועילה לגמרי, ולא אומרת אותה. זו התשובה שמקבלים כשהאחזור החמיץ בשקט את הקטע שהחזיק את המספר.

</div>

In [ ]:
example = raw[0]
print("question:", example["question"])
print("fact    :", example["fact"], "\n")
print("KEPT:", example["kept"][:150])
print("LOST:", example["lost"][:150])
print("\nground truth, decided without any model:")
print("  kept contains the fact:", contains_fact(example["kept"], example["fact"]))
print("  lost contains the fact:", contains_fact(example["lost"], example["fact"]))

<div dir="rtl" lang="he" align="right" markdown="1">

## השופט הראשון: ציון מ-1 עד 5

זו השאלה שכולם שואלים: "דרג את התשובה מ-1 עד 5". נריץ אותה על כל 24 התשובות ונראה מה היא אומרת.

התשובות של המודל הוקלטו מראש, כך שהפרק רץ בלי אינטרנט ובלי עלות. המודל הוא קטן — `3B` — ועל כך בסוף הפרק.

</div>

In [ ]:
ask = asker(cassettes=CASSETTES, model="Llama-3.2-3B-Instruct-Q4_K_M",
            temperature=0.0, max_tokens=64, seed=7)
rows = judge_cases(cases, ask)

h = holistic_summary(rows)
print(f"mean score                     {h['mean']:.2f}")
print(f"called the reply good          {h['called_good']:.1%}")
print(f"called it good when fact lost  {h['called_good_when_fact_lost']:.1%}")

In [ ]:
kept = [r["holistic"].value for r in rows if r["truth"] and r["holistic"].decided]
lost = [r["holistic"].value for r in rows if not r["truth"] and r["holistic"].decided]
print(f"answers that KEPT the fact: mean {mean(kept):.2f}")
print(f"answers that LOST the fact: mean {mean(lost):.2f}")
print(f"the gap                   : {mean(kept) - mean(lost):+.2f}  on a scale of five")

<div dir="rtl" lang="he" align="right" markdown="1">

## המספר שסוגר את העניין

ממוצע הוא עדיין מדד רך. השאלה החדה היא אחרת: לכל שאלה יש בדיוק שתי תשובות, אחת טובה ואחת שאיבדה את העובדה. בכמה מהמקרים השופט נתן לטובה ציון גבוה יותר?

זו שאלה שיש לה תשובה אחת נכונה מתוך שתיים, ולכן יש לה גם רף ברור: מטבע מקבל חצי.

</div>

In [ ]:
pairs = {}
for r in rows:
    pairs.setdefault(r["question"], {})[r["variant"]] = r

separated_h = sum(1 for p in pairs.values()
                  if p["kept"]["holistic"].decided and p["lost"]["holistic"].decided
                  and p["kept"]["holistic"].value > p["lost"]["holistic"].value)
separated_b = sum(1 for p in pairs.values()
                  if p["kept"]["binary"].value and not p["lost"]["binary"].value)

print(f"holistic judge separated the pair   {separated_h}/{len(pairs)}")
print(f"binary judge separated the pair     {separated_b}/{len(pairs)}")
print(f"a coin would get                    {len(pairs)//2}/{len(pairs)}")

<div dir="rtl" lang="he" align="right" markdown="1">

## השופט השני: שאלה אחת שאפשר להכריע

עכשיו נחליף את השאלה. במקום "עד כמה התשובה טובה", נשאל שאלה אחת צרה שיש לה תשובה של כן או לא: "האם התשובה אומרת שהטוקן תקף שישים דקות".

זה לא פותר הכל, אבל זה משנה את סוג הטעות — ושני סוגי הטעויות צריכים להימדד בנפרד, כי הם דורשים תיקון שונה לגמרי.

</div>

In [ ]:
b = agreement([r["binary"] for r in rows], [r["truth"] for r in rows])
print(f"accuracy         {b['accuracy']:.1%}   (n={b['n']}, undecided={b['undecided']})")
print(f"said yes         {b['said_yes']:.1%}")
print(f"false positive   {b['false_positive']:.1%}   <- approved an answer that lost the fact")
print(f"false negative   {b['false_negative']:.1%}   <- rejected an answer that kept it")

<div dir="rtl" lang="he" align="right" markdown="1">

## והשופט השלישי, שאין בו מודל בכלל

יש אפשרות שלישית שקל לשכוח: לבדוק אם העובדה פשוט מופיעה בתשובה. זו בדיקת מחרוזת. היא חינמית, מיידית, ואי אפשר להחניף לה.

היא גם, בפרק הזה, צדקה בכל 24 המקרים — כי היא זו שהגדירה מלכתחילה מה נכון. זה נשמע כמו רמאות, ובדיוק זו הנקודה: אם אפשר להכריע שאלה בלי מודל, אין שום סיבה לשלם למודל שיכריע אותה פחות טוב.

</div>

In [ ]:
# The string check agrees with ground truth on every case - because it *is* the ground
# truth. That is not cheating; it is the point. A question you can settle without a model
# should not be handed to one.
model_judge = sum(1 for r in rows if r["binary"].value is r["truth"])
print(f"agreed with ground truth: binary judge {model_judge}/{len(rows)}, "
      f"string check {len(rows)}/{len(rows)}")

viz.climb({"holistic": separated_h / len(pairs),
           "binary": separated_b / len(pairs),
           "no model at all": 1.0},
          label="pairs separated correctly",
          title="which judge can tell the two answers apart");

<div dir="rtl" lang="he" align="right" markdown="1">

## מה לוקחים מכאן

**מדוד את שני החצאים בנפרד.** האחזור נמדד בחשבון מדויק, והוא החצי שבדרך כלל אשם. מי שמודד רק את התשובה הסופית לא יֵדע מה לתקן.

**שאלה רחבה לשופט מחזירה תשובה חסרת ערך.** "עד כמה זה טוב" קיבל כאן `3.83` מול `3.42` — פער של פחות מחצי נקודה בין תשובה נכונה לתשובה שאיבדה את העיקר.

**דיוק לבדו משקר.** השופט הצר טעה לשני הכיוונים בשיעורים שונים מאוד, ושני הסוגים דורשים תיקון אחר. מספר אחד היה מסתיר את זה.

**אם אפשר לבדוק בלי מודל, בדקו בלי מודל.** זה חינם, זה מיידי, וזה לא מתווכח.

**והסתייגות על המספרים שכאן.** השופט בפרק הוא מודל בגודל `3B` שרץ על המעבד, וזו הסיבה שהוא חלש כל כך. שופט גדול יותר יצליח יותר. מה שלא משתנה הוא השיטה: למדוד את השופט מול תשובות שידועות מראש, לפני שסומכים עליו. שופט שלא נמדד הוא לא מדידה — הוא דעה נוספת.

</div>

In [ ]:
print(f"holistic judge: {separated_h}/{len(pairs)} pairs separated  (a coin gets {len(pairs)//2})")
print(f"binary judge  : {separated_b}/{len(pairs)}")
print(f"string check  : {len(pairs)}/{len(pairs)}, for free")